In [ ]:
%%sql -r ctx
USE DATABASE WIND_TURBINE_POC;
USE SCHEMA STAGING;
USE WAREHOUSE WIND_TURBINE_POC_WH;
USE ROLE POC_ETL_ROLE;

In [ ]:
from snowflake.snowpark_connect import init_spark_session
from pyspark.sql import functions as F

spark = init_spark_session()
spark.conf.set("spark.sql.session.timeZone", "UTC")
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
-- Get files list from Stage STAGING.WIND_TURBINE
SELECT
    RELATIVE_PATH   AS FILE_NAME,
    LAST_MODIFIED   AS FILE_LAST_MODIFIED,
    MD5             AS FILE_CHECKSUM,
    SIZE            AS FILE_SIZE
FROM DIRECTORY(@WIND_TURBINE_POC.STAGING.WIND_TURBINE)

In [ ]:
# Save stage file listing into a temp table for MERGE
stage_files_df.write.mode("overwrite").save_as_table(
    "WIND_TURBINE_POC.STAGING.STAGE_FILES_TEMP", table_type="temporary"
)

# MERGE: insert new files, update changed files (only completed) or not loaded files
merge_result = session.sql("""
    MERGE INTO WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY AS target
    USING WIND_TURBINE_POC.STAGING.STAGE_FILES_TEMP AS source
    ON target.FILE_NAME = source.FILE_NAME
    WHEN MATCHED
        AND (target.FILE_LAST_MODIFIED < source.FILE_LAST_MODIFIED
             AND target.FILE_CHECKSUM != source.FILE_CHECKSUM
             AND target.STATUS = 'COMPLETED')
        OR STATUS = 'PROCESSING'
    THEN UPDATE SET
        target.FILE_LAST_MODIFIED   = source.FILE_LAST_MODIFIED,
        target.FILE_CHECKSUM        = source.FILE_CHECKSUM,
        target.FILE_SIZE            = source.FILE_SIZE,
        target.STATUS_DATETIME      = CONVERT_TIMEZONE('UTC', CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        target.STATUS               = 'PROCESSING'
    WHEN NOT MATCHED THEN INSERT (
        FILE_NAME,
        FILE_LAST_MODIFIED,
        FILE_CHECKSUM,
        FILE_SIZE,
        STATUS,
        STATUS_DATETIME
    ) VALUES (
        source.FILE_NAME,
        source.FILE_LAST_MODIFIED,
        source.FILE_CHECKSUM,
        source.FILE_SIZE,
        'PROCESSING',
        CONVERT_TIMEZONE('UTC', CURRENT_TIMESTAMP())::TIMESTAMP_NTZ
    )
""").collect()

print(f"MERGE result: {merge_result}")

session.table("WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY").show()

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame

processing_files = [
    row["FILE_NAME"]
    for row in session.table("WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY")
        .filter("STATUS = 'PROCESSING'")
        .select("FILE_NAME")
        .collect()
]

if not processing_files:
    print("No files with STATUS 'PROCESSING' to load.")
else:
    print(f"Loading {len(processing_files)} file(s): {processing_files}")

    for fname in processing_files:
        file_df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "false")
            .load(f"@WIND_TURBINE_POC.STAGING.WIND_TURBINE/{fname}")
            .withColumnRenamed("timestamp", "MEASUREMENT_TIMESTAMP")
            .withColumnRenamed("turbine_id", "TURBINE_ID")
            .withColumnRenamed("wind_speed", "WIND_SPEED")
            .withColumnRenamed("wind_direction", "WIND_DIRECTION")
            .withColumnRenamed("power_output", "POWER_OUTPUT")
            .withColumn("SOURCE_FILE", F.lit(fname))
            .withColumn("LOADED_DATETIME", F.current_timestamp())
        )

        (   file_df
            .select(
            "MEASUREMENT_TIMESTAMP",
            "TURBINE_ID",
            "WIND_SPEED",
            "WIND_DIRECTION",
            "POWER_OUTPUT",
            "SOURCE_FILE",
            "LOADED_DATETIME"
            )
            .write
            .mode("append")
            .saveAsTable(
                "WIND_TURBINE_POC.STAGING.STG_TURBINE_MEASUREMENT"
            )
        )    
        print(f"File {fname} file is loaded.")

    -- Update file status from PROCESSING to LOADED
    update_result = session.sql("""
    UPDATE WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY
    SET
        STATUS = 'LOADED'
    WHERE STATUS = 'PROCESSING'
    """).collect()

    print(f"Update result: {update_result}")

    session.table("WIND_TURBINE_POC.STAGING.FILE_LOAD_HISTORY").show()